In [1]:
print("hello")

hello


In [2]:
"""
Script to test SAE and explain document rankings using concept activations.

This script:
1. Takes a query and gets top 1000 documents (as in 3_test_biencoder_moe.py)
2. Gets embeddings for query and documents
3. Verifies the ranking matches the original ranking
4. Uses SAE to get concept activations for query and documents
5. Highlights which concepts are most activated by the query
6. Displays concepts most activated by relevant documents

Usage:
    # Test with a specific query ID
    python test_sae_explanations.py query_id=12345 top_k_concepts=15
    
    # Test with random query (will pick first query with relevant docs)
    python test_sae_explanations.py top_k_concepts=10
    
    # With dataset/model overrides
    python test_sae_explanations.py query_id=12345 dataset=msmarco model.init.save_model=bert
"""

import json
import logging
import os

import hydra
import torch
import numpy as np
import tqdm
from indxr import Indxr
from omegaconf import DictConfig, OmegaConf
from transformers import AutoModel, AutoTokenizer, AutoConfig

from model.models import MoEBiEncoder
from model.utils import seed_everything
from overcomplete.overcomplete.sae import TopKRAESAE

from ranx import Qrels

logger = logging.getLogger(__name__)


# def load_sae_model(cfg, device):
#     """Load the trained SAE model."""
#     # Determine number of concepts from config or use default
#     embedding_dim = cfg.model.init.embedding_size
#     nb_concepts = cfg.sae.nb_concepts if hasattr(cfg, 'sae') and hasattr(cfg.sae, 'nb_concepts') else embedding_dim * 2
    
#     # Load archetypal points
#     points_path = os.path.join(
#         cfg.dataset.model_dir,
#         f"sae_archetypal_points_{cfg.model.init.save_model}_experts{cfg.model.adapters.num_experts}_concepts{nb_concepts}.pt"
#     )
    
#     if not os.path.exists(points_path):
#         raise FileNotFoundError(f"Archetypal points not found at {points_path}")
    
#     archetypal_points = torch.load(points_path, map_location=device)
#     logger.info(f"Loaded archetypal points from {points_path}")
    
#     # Get SAE config parameters
#     top_k = cfg.sae.top_k if hasattr(cfg, 'sae') and hasattr(cfg.sae, 'top_k') and cfg.sae.top_k is not None else None
#     encoder_module = cfg.sae.encoder_module if hasattr(cfg, 'sae') and hasattr(cfg.sae, 'encoder_module') and cfg.sae.encoder_module is not None else None
#     delta = cfg.sae.delta if hasattr(cfg, 'sae') and hasattr(cfg.sae, 'delta') else 1.0
    
#     # Create SAE model
#     sae_model = TopKRAESAE(
#         input_shape=embedding_dim,
#         nb_concepts=nb_concepts,
#         points=archetypal_points,
#         top_k=top_k,
#         encoder_module=encoder_module,
#         delta=delta,
#         device=device
#     )
    
#     # Load trained weights
#     model_path = os.path.join(
#         cfg.dataset.model_dir,
#         f"sae_{cfg.model.init.save_model}_experts{cfg.model.adapters.num_experts}_concepts{nb_concepts}.pt"
#     )
    
#     if not os.path.exists(model_path):
#         raise FileNotFoundError(f"SAE model not found at {model_path}")
    
#     sae_model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
#     sae_model = sae_model.to(device)
#     sae_model.eval()
#     logger.info(f"Loaded SAE model from {model_path}")
    
#     return sae_model, nb_concepts


# def get_top_concepts(activations, top_k=10):
#     """Get top-k most activated concepts from activation vector."""
#     if isinstance(activations, torch.Tensor):
#         activations = activations.cpu().numpy()
    
#     # Flatten if needed
#     if len(activations.shape) > 1:
#         activations = activations.flatten()
    
#     # Get top-k indices
#     top_indices = np.argsort(activations)[::-1][:top_k]
#     top_values = activations[top_indices]
    
#     return list(zip(top_indices, top_values))


# def verify_ranking(query_embedding, doc_embeddings, original_scores, top_k=1000):
#     """Verify that the ranking using embeddings matches the original ranking."""
#     # Compute similarity scores using embeddings
#     scores = torch.einsum('xy, ly -> x', doc_embeddings, query_embedding)
#     scores = scores.cpu().numpy()
    
#     # Get sorted indices (descending order)
#     sorted_indices = np.argsort(scores)[::-1]
    
#     # Original ranking should be [0, 1, 2, ..., top_k-1] since docs are already sorted
#     # But we compare the actual score ordering
#     original_sorted_indices = np.argsort(original_scores)[::-1]
    
#     # Check if rankings match (at least for top documents)
#     # Compare top 100 positions
#     check_top = min(100, len(sorted_indices), len(original_sorted_indices))
#     matches = 0
#     for i in range(check_top):
#         # Check if the document at position i in new ranking is within position i±5 in original
#         new_doc_idx = sorted_indices[i]
#         # Find where this doc appears in original ranking
#         original_pos = np.where(original_sorted_indices == new_doc_idx)[0]
#         if len(original_pos) > 0:
#             original_pos = original_pos[0]
#             # Allow some flexibility: within 5 positions
#             if abs(original_pos - i) <= 5:
#                 matches += 1
    
#     match_rate = matches / check_top if check_top > 0 else 0.0
    
#     # Also compute correlation between scores
#     score_correlation = np.corrcoef(scores, original_scores)[0, 1] if len(scores) == len(original_scores) else 0.0
    
#     return match_rate, score_correlation, scores, sorted_indices


# @hydra.main(config_path="../conf", config_name="config", version_base=None)
# def main(cfg: DictConfig):
#     # Get query_id from config override or use None to pick random
#     # Can be set via: python test_sae_explanations.py query_id=12345 top_k_concepts=15
#     query_id = getattr(cfg, 'query_id', None) if hasattr(cfg, 'query_id') else None
#     top_k_concepts = getattr(cfg, 'top_k_concepts', 10) if hasattr(cfg, 'top_k_concepts') else 10
    
#     os.makedirs(cfg.dataset.output_dir, exist_ok=True)
#     os.makedirs(cfg.dataset.logs_dir, exist_ok=True)
#     os.makedirs(cfg.dataset.model_dir, exist_ok=True)
    
#     logging_file = f"{cfg.model.init.doc_model.replace('/','_')}_test_sae_explanations.log"
#     logging.basicConfig(
#         filename=os.path.join(cfg.dataset.logs_dir, logging_file),
#         filemode='a',
#         format='%(asctime)s,%(msecs)d %(name)s %(levelname)s %(message)s',
#         datefmt='%Y-%m-%d %H:%M-%S',
#         level=logging.INFO
#     )
    
#     # Also print to console
#     console_handler = logging.StreamHandler()
#     console_handler.setLevel(logging.INFO)
#     logger.addHandler(console_handler)
    
#     seed_everything(cfg.general.seed)
    
#     device = cfg.model.init.device
    
#     # Load bi-encoder model
#     logger.info("Loading bi-encoder model...")
#     tokenizer = AutoTokenizer.from_pretrained(cfg.model.init.tokenizer)
#     config = AutoConfig.from_pretrained(cfg.model.init.doc_model)
#     config.num_experts = cfg.model.adapters.num_experts
#     config.adapter_residual = cfg.model.adapters.residual
#     config.adapter_latent_size = cfg.model.adapters.latent_size
#     config.adapter_non_linearity = cfg.model.adapters.non_linearity
#     config.use_adapters = cfg.model.adapters.use_adapters
#     doc_model = AutoModel.from_pretrained(cfg.model.init.doc_model)
#     model = MoEBiEncoder(
#         doc_model=doc_model,
#         tokenizer=tokenizer,
#         num_classes=cfg.model.adapters.num_experts,
#         normalize=cfg.model.init.normalize,
#         specialized_mode=cfg.model.init.specialized_mode,
#         pooling_mode=cfg.model.init.aggregation_mode,
#         use_adapters=cfg.model.adapters.use_adapters,
#         device=device
#     )
    
#     # Load model weights
#     # if cfg.model.adapters.use_adapters:
#     #     if cfg.model.init.specialized_mode == "sbmoe_top1":
#     #         model_path = f'{cfg.dataset.model_dir}/{cfg.model.init.save_model}_experts3-variant_top1.pt'
#     #     elif cfg.model.init.specialized_mode == "sbmoe_all":
#     #         model_path = f'{cfg.dataset.model_dir}/{cfg.model.init.save_model}_experts{cfg.model.adapters.num_experts}-variant_top1.pt'
#     #     elif cfg.model.init.specialized_mode == "random":
#     #         model_path = f'{cfg.dataset.model_dir}/{cfg.model.init.save_model}_experts{cfg.model.adapters.num_experts}-random.pt'
#     # else:
#     model_path = f'{cfg.dataset.model_dir}/{cfg.model.init.save_model}_experts{cfg.model.adapters.num_experts}-ft.pt'
    
#     model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
#     model = model.to(device)
#     model.eval()
#     logger.info(f"Loaded bi-encoder model from {model_path}")
    
#     # Load document embeddings
#     logger.info("Loading document embeddings...")
#     doc_embedding = torch.load(
#         f'{cfg.testing.embedding_dir}/{cfg.model.init.save_model}_experts{cfg.model.adapters.num_experts}_fullrank.pt',
#         map_location=device,
#         weights_only=True
#     ).to(device)
    
#     with open(f'{cfg.testing.embedding_dir}/id_to_index_{cfg.model.init.save_model}_experts{cfg.model.adapters.num_experts}_fullrank.json', 'r') as f:
#         id_to_index = json.load(f)
    
#     index_to_id = {ind: _id for _id, ind in id_to_index.items()}
    
#     # Load queries and qrels
#     data = Indxr(cfg.testing.query_path, key_id='_id')
#     ranx_qrels = Qrels.from_file(cfg.testing.qrels_path)
    
#     # Load SAE model
#     logger.info("Loading SAE model...")
#     sae_model, nb_concepts = load_sae_model(cfg, device)
    
#     # Get query
#     if query_id:
#         query_id = query_id
#     else:
#         # Get a random query that has relevant documents
#         query_ids = list(data.keys())
#         query_id = None
#         for qid in query_ids[:100]:  # Check first 100 queries
#             if qid in ranx_qrels.qrels and len(ranx_qrels.qrels[qid]) > 0:
#                 query_id = qid
#                 break
        
#         if query_id is None:
#             logger.error("No query with relevant documents found in first 100 queries")
#             return
    
#     query_data = data.get(query_id)
#     if query_data is None:
#         logger.error(f"Query ID {query_id} not found in data")
#         return
    
#     logger.info(f"Testing query ID: {query_id}")
#     logger.info(f"Query text: {query_data['text']}")
    
#     # Get query embedding
#     with torch.no_grad():
#         query_embedding = model.query_encoder([query_data['text']]).to(device)
    
#     # Get top 1000 documents (as in 3_test_biencoder_moe.py)
#     logger.info("Computing document rankings...")
#     bert_scores = torch.einsum('xy, ly -> x', doc_embedding, query_embedding)
#     index_sorted = torch.argsort(bert_scores, descending=True)
#     top_k_indices = index_sorted[:1000]
#     top_k_ids = [index_to_id[int(_id)] for _id in top_k_indices]
#     top_k_scores = bert_scores[top_k_indices].cpu().numpy()
    
#     logger.info(f"Retrieved top 1000 documents")
    
#     # Get document embeddings for top 1000
#     top_doc_embeddings = doc_embedding[top_k_indices]
    
#     # Verify ranking
#     logger.info("Verifying ranking consistency...")
#     match_rate, score_correlation, recomputed_scores, sorted_indices = verify_ranking(
#         query_embedding, top_doc_embeddings, top_k_scores, top_k=1000
#     )
#     logger.info(f"Ranking match rate (top 100, ±5 positions): {match_rate:.2%}")
#     logger.info(f"Score correlation: {score_correlation:.4f}")
    
#     # Get relevant documents
#     relevants_dict = ranx_qrels.to_dict()
#     relevants = set(relevants_dict.get(query_id, {}).keys())
#     relevant_indices_in_topk = [
#         i for i, doc_id in enumerate(top_k_ids) if doc_id in relevants
#     ]
    
#     logger.info(f"Found {len(relevant_indices_in_topk)} relevant documents in top 1000")
    
#     # Get concept activations for query
#     logger.info("Computing concept activations...")
#     with torch.no_grad():
#         query_pre_codes, query_codes = sae_model.encode(query_embedding)
#         query_activations = query_codes[0].cpu().numpy()  # Get first (and only) query
    
#     # Get concept activations for top documents
#     with torch.no_grad():
#         doc_pre_codes, doc_codes = sae_model.encode(top_doc_embeddings)
#         doc_activations = doc_codes.cpu().numpy()
    
#     # Get top concepts for query
#     top_query_concepts = get_top_concepts(query_activations, top_k=top_k_concepts)
    
#     logger.info("\n" + "="*80)
#     logger.info("QUERY CONCEPT ACTIVATIONS")
#     logger.info("="*80)
#     logger.info(f"Query: {query_data['text']}")
#     logger.info(f"\nTop {top_k_concepts} concepts activated by query:")
#     for i, (concept_idx, activation_value) in enumerate(top_query_concepts, 1):
#         logger.info(f"  {i}. Concept {concept_idx}: {activation_value:.6f}")
    
#     # Get top concepts for relevant documents
#     if relevant_indices_in_topk:
#         logger.info("\n" + "="*80)
#         logger.info("RELEVANT DOCUMENTS CONCEPT ACTIVATIONS")
#         logger.info("="*80)
        
#         # Average activations across relevant documents
#         relevant_doc_activations = doc_activations[relevant_indices_in_topk]
#         avg_relevant_activations = np.mean(relevant_doc_activations, axis=0)
        
#         top_relevant_concepts = get_top_concepts(avg_relevant_activations, top_k=args.top_k_concepts)
        
#         logger.info(f"\nAverage activations across {len(relevant_indices_in_topk)} relevant documents:")
#         for i, (concept_idx, activation_value) in enumerate(top_relevant_concepts, 1):
#             logger.info(f"  {i}. Concept {concept_idx}: {activation_value:.6f}")
        
#         # Show individual relevant documents
#         logger.info("\n" + "-"*80)
#         logger.info("Individual Relevant Documents:")
#         logger.info("-"*80)
        
#         for idx in relevant_indices_in_topk[:10]:  # Show first 10 relevant docs
#             doc_id = top_k_ids[idx]
#             rank = idx + 1
#             score = top_k_scores[idx]
            
#             doc_top_concepts = get_top_concepts(doc_activations[idx], top_k=5)
            
#             logger.info(f"\nRank {rank} | Doc ID: {doc_id} | Score: {score:.6f}")
#             logger.info(f"Top 5 concepts:")
#             for i, (concept_idx, activation_value) in enumerate(doc_top_concepts, 1):
#                 logger.info(f"  {i}. Concept {concept_idx}: {activation_value:.6f}")
        
#         # Compare query and relevant document concepts
#         logger.info("\n" + "="*80)
#         logger.info("CONCEPT OVERLAP ANALYSIS")
#         logger.info("="*80)
        
#         query_top_concept_set = set([idx for idx, _ in top_query_concepts])
#         relevant_top_concept_set = set([idx for idx, _ in top_relevant_concepts])
        
#         overlap = query_top_concept_set.intersection(relevant_top_concept_set)
        
#         logger.info(f"Query top {top_k_concepts} concepts: {sorted(query_top_concept_set)}")
#         logger.info(f"Relevant docs top {top_k_concepts} concepts: {sorted(relevant_top_concept_set)}")
#         logger.info(f"Overlapping concepts: {sorted(overlap)} ({len(overlap)}/{top_k_concepts})")
        
#     else:
#         logger.warning("No relevant documents found in top 1000")
    
#     logger.info("\n" + "="*80)
#     logger.info("Analysis complete!")
#     logger.info("="*80)


# if __name__ == '__main__':
#     main()


ModuleNotFoundError: No module named 'overcomplete.sae'

In [ ]:
# Get query_id from config override or use None to pick random
    # Can be set via: python test_sae_explanations.py query_id=12345 top_k_concepts=15
query_id = getattr(cfg, 'query_id', None) if hasattr(cfg, 'query_id') else None
top_k_concepts = getattr(cfg, 'top_k_concepts', 10) if hasattr(cfg, 'top_k_concepts') else 10